# Plot regional mortality due to exposure to PM<sub>2.5</sub> as a box plot

Using one ensemble member as an example.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from utils.utils import get_scenario_config

In [ ]:
def process_mortality_per_100k(da):
    # === Path config ===
    MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
    POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

    # === Load data ===
    mask_file = "GBD_Region_Masks_0.10.nc"
    mask_path = os.path.join(MASK_DIR, mask_file)
    mask = xr.open_dataarray(mask_path)

    # Regional population
    pop_file = "ssp2_region_level_2000-2100.nc"
    pop_path = os.path.join(POP_DIR, pop_file)
    population = xr.open_dataarray(pop_path)
    pop = population.reindex_like(mask, method="nearest", tolerance=1e-9)

    # Calculate end of scenario mean (10 years)
    end_idx = da.sizes["year"]
    # Select the last 10 years
    da_last = da.isel(year=slice(end_idx - 10, end_idx))

    # Get the first and last time values
    start_time = da_last.year[0].item()
    end_time = da_last.year[-1].item()

    # Calculate temporal mean
    da_mean = da_last.mean("year")

    # Calculate population mean
    years = slice(start_time, end_time)
    av_pop = pop.sel(year=years).mean("year")

    # Calculate mortality per 100,000
    da_per_100k = (da_mean/av_pop)*100000
    return da_per_100k, start_time, end_time

In [ ]:
# === Path Config ===
SAVE_DIR = "/glade/u/home/awells/air_quality_project/plotting/example_workflow/"
model = "CESM2"
scenario = "SSP245"
config = get_scenario_config(model, scenario)
years = config["years"]
dates = f"{years.start}-{years.stop}"

n_samples = 200

# Load data
DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/"
ens1_file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_01_{dates}.nc"
ens1_path = os.path.join(DIR, ens1_file)
ens1 = xr.open_dataarray(ens1_path)

In [ ]:
# Final 10 years per 100k
da_mean, first_year, last_year = process_mortality_per_100k(ens1)

In [ ]:
# Sort regions by mean value
region_means = da_mean.mean(dim="samples")
sorted_regions = region_means.sortby(region_means).region.values

data = [da_mean.sel(region=r).values for r in sorted_regions]

fcolor = "#B73779"

# --- Plot ---
plt.figure(figsize=(8, 10))
plt.boxplot(data, vert=False, patch_artist=True, sym="+",
            widths=0.6,
            boxprops=dict(facecolor=fcolor, color="gray", lw=0.5),
            medianprops=dict(color="white", lw=1),
            whiskerprops=dict(color="gray"),
            capprops=dict(color="gray"),
            flierprops=dict(color="gray", alpha=0.4))

plt.yticks(np.arange(1, len(sorted_regions) + 1), sorted_regions)
plt.xlabel("Total PM$_{2.5}$ Mortality per 100,000 (year$^{-1}$)")
plt.title(f"{model} SSP2-4.5 ensemble 1\n {first_year}-{last_year}")
plt.grid(axis="y", linestyle=':', alpha=0.5)
plt.grid(axis='x', linestyle='--', alpha=0.8)

# --- Symmetric log scale with real-number tick labels ---
plt.xscale('symlog', linthresh=1)

# Use ScalarFormatter to show tick labels as real numbers
formatter = ScalarFormatter()
formatter.set_scientific(False)
plt.gca().xaxis.set_major_formatter(formatter)

out_file = f"Mortality_per_100k_pm25_region_boxplot_{model}_{scenario}_{first_year}-{last_year}.png"
out_path = os.path.join(SAVE_DIR, out_file)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.tick_params(axis="y", length=0)

plt.tight_layout()
plt.savefig(out_path)
plt.show()